step1:to load the data from landing to bronze layer


In [0]:
create streaming table orders_bronze as 
select *,_metadata.file_path as file_name,current_timestamp() as load_time
from cloud_files('/Volumes/workspace/default/orders_datasets/orders/','csv',
map('cloud_files.inferColumnTypes','True'))

step 2:to load into silver table cleaned record

In [0]:
create streaming table orders_silver(
  constraint order_id_con expect(order_id is not null) on violation drop row
)as 
select * from stream(live.orders_bronze);

step 2: using merge to aviod duplicate

In [0]:
create streaming table orders_silver_cleaned;
apply changes into live.orders_silver_cleaned
from stream(live.orders_silver)
keys(order_id)
sequence by load_time
stored as  SCD TYPE 2;

step 3:load the data into gold layer create materialized view


In [0]:
create materialized view  orders_gold_complete
as 
select * from live.orders_silver_cleaned
where order_status='COMPLETE'